# G1: Correlation-Dimension Revalidation of Rollout-Horizon Claims

**Scope (narrowed, per Section 12's own correction):** whether long-horizon
autoregressive rollout forecasts (H>128, chained passes) preserve correct
attractor structure. Pointwise MAE cannot answer this; correlation dimension
(Grassberger-Procaccia), matching the published paper's own distributional
method, can. Targets: the Rossler significance result and the Burgers/
Harmonic OOD rows at H=336 (Section 7, needs `baseline_100k`/
`ablation_100k`), plus Double Pendulum and Lorenz rho=10 (early benchmark,
published checkpoints, flagged for MAE/Hellinger disagreement).

**Hard gate, per Section 1.2's estimator-validation rule.** This is new
instrumentation, not a rerun of existing code. Two prior estimators in this
project failed this exact rule (Rosenstein lambda1, twice; naive correlation
dimension, which returned d=0.86 on Lorenz against literature ~2.05 and was
retired). The one estimator that worked (persistent homology, Section 10)
took three iterations against known-dimension test signals before being
trusted on real data. **Part 1 below must pass its gates before Part 2 runs
on anything real.** If gates fail, this notebook stops and reports the
failure -- it does not proceed anyway.

**Checkpoint loading note.** The `baseline_100k`/`ablation_100k` loading
cell below is built from this project's documented convention
(`PatchTSTForPrediction` + `load_patchtst_model`, architecture from
training-time config, strict `state_dict` load), not from a pasted source
cell -- that source was not available when this notebook was built. A hard
assertion on `training_info.json`'s `use_dynamics_embedding` field is
included as a safety check (baseline=True, ablation=False); if your actual
checkpoint structure differs, this should fail loudly here rather than
silently loading the wrong checkpoint into the wrong slot. **Fill in
CHECKPOINT_DIR below before running Part 2b.**


In [1]:
# ============================================================
# CELL 1 -- IMPORTS + MODEL LOADING (both paths)
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
from scipy.integrate import solve_ivp
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

import sys
sys.path.insert(0, './panda')

# --- Path A: published checkpoints (Part 2a: Double Pendulum, Lorenz rho=10) ---
from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model_published = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)
chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)
print("Published checkpoints loaded.")

# --- Path B: retrained baseline_100k/ablation_100k (Part 2b: Rossler, Burgers/Harmonic) ---
# Direct paths to the weight files -- no shared parent directory. training_info.json
# is looked up in each file's own directory (os.path.dirname), matching the convention
# that it sits alongside the weights, not by reconstructing a directory + fixed
# subpath as the earlier draft of this cell assumed.
BASELINE_CKPT_PATH = "C:/Users/user/Downloads/panda-100k-baseline-checkpoint/model.safetensors"
ABLATION_CKPT_PATH = "C:/Users/user/Downloads/panda-100k-ablation-checkpoint/model.safetensors"

import json as _json
from panda.patchtst.patchtst import PatchTSTForPrediction
from panda.utils.train_utils import load_patchtst_model

# Verbatim from user-supplied source. use_dynamics_embedding is the only field
# that varies between baseline and ablation.
def _build_model_config(use_dynamics_embedding):
    return dict(
        mode='predict',
        context_length=512,
        prediction_length=128,
        patch_length=16,
        patch_stride=16,
        num_hidden_layers=8,
        d_model=512,
        num_attention_heads=8,
        channel_attention=True,
        ffn_dim=512,
        norm_type='rmsnorm',
        norm_eps=1e-5,
        attention_dropout=0.0,
        positional_dropout=0.0,
        path_dropout=0.0,
        ff_dropout=0.0,
        bias=True,
        activation_function='gelu',
        pre_norm=True,
        use_cls_token=False,
        init_std=0.02,
        scaling='std',
        pooling_type='max',
        head_dropout=0.0,
        channel_rope=False,
        max_wavelength=500,
        rope_percent=0.75,
        loss='mse',
        distribution_output=None,
        use_dynamics_embedding=use_dynamics_embedding,
        num_poly_feats=120,
        poly_degrees=2,
        rff_trainable=False,
        rff_scale=1.0,
        num_rff=256,
        do_mask_input=None,
        mask_type='random',
        random_mask_ratio=0.5,
        channel_consistent_masking=False,
        mask_value=0,
        num_forecast_mask_patches=3,
        unmasked_channel_indices=None,
        num_parallel_samples=100,
    )

def load_retrained_checkpoint(weights_path, expected_use_dynamics_embedding, arm_name):
    """Two-stage process, per user-supplied source: load_patchtst_model builds a
    FRESH untrained architecture from a config dict; weights are loaded separately
    from the EXACT file path given (not reconstructed from a directory + assumed
    filename); the raw model is then wrapped in a PatchTSTPipeline, since that's
    the interface panda_forecast_traj actually calls (.predict(...)), not the raw
    PatchTSTForPrediction model.

    weights_path: exact path to model.safetensors or pytorch_model.bin.
    training_info.json is looked up in the SAME directory as weights_path."""
    ckpt_dir = os.path.dirname(weights_path)
    info_path = os.path.join(ckpt_dir, "training_info.json")
    if os.path.exists(info_path):
        with open(info_path) as f:
            info = _json.load(f)
        actual = info.get("use_dynamics_embedding")
        assert actual == expected_use_dynamics_embedding, (
            f"ARM MISMATCH at {weights_path}: training_info.json says "
            f"use_dynamics_embedding={actual}, expected {expected_use_dynamics_embedding}. "
            "Wrong checkpoint attached -- stopping before use."
        )
        print(f"Checkpoint identity verified: run_name={info.get('run_name')}, "
              f"use_dynamics_embedding={actual}")
    else:
        print(f"[WARNING] training_info.json missing at {info_path} -- arm identity "
              f"NOT auto-verified. Confirm manually before trusting {arm_name} results.")

    model_config = _build_model_config(expected_use_dynamics_embedding)
    model = load_patchtst_model(
        mode='predict',
        model_config=model_config,
        pretrained_encoder_path=None,
        pretained_checkpoint=None,
    )

    assert os.path.exists(weights_path), f"Weights file not found: {weights_path}"
    if weights_path.endswith(".safetensors"):
        from safetensors.torch import load_file as _load_sf
        state = _load_sf(weights_path)
    elif weights_path.endswith(".bin"):
        state = torch.load(weights_path, map_location='cpu')
    else:
        raise ValueError(f"Unrecognized weights file extension: {weights_path}")

    model.load_state_dict(state, strict=True)
    model = model.to(device).eval()  # TODO: confirm PatchTSTPipeline doesn't need this
                                       # done differently -- added defensively since the
                                       # source snippet was from a training context, not
                                       # inference, and didn't show device placement.
    pipeline = PatchTSTPipeline(mode='predict', model=model)
    print(f"Loaded and wrapped {arm_name}: {weights_path}")
    return pipeline

baseline_100k = None
ablation_100k = None
if BASELINE_CKPT_PATH and ABLATION_CKPT_PATH:
    try:
        baseline_100k = load_retrained_checkpoint(BASELINE_CKPT_PATH, expected_use_dynamics_embedding=True, arm_name="baseline_100k")
        ablation_100k = load_retrained_checkpoint(ABLATION_CKPT_PATH, expected_use_dynamics_embedding=False, arm_name="ablation_100k")
    except Exception as e:
        print(f"[WARNING] Retrained checkpoint loading failed: {e}")
        print("Part 2b will be skipped. Part 1 (gates) and Part 2a (published checkpoints) unaffected.")
else:
    print("[INFO] Checkpoint paths not set -- Part 2b (Rossler, Burgers/Harmonic) will be skipped "
          "until they are filled in.")


Device: cpu
Published checkpoints loaded.
Checkpoint identity verified: run_name=baseline, use_dynamics_embedding=True
Loaded and wrapped baseline_100k: C:/Users/user/Downloads/panda-100k-baseline-checkpoint/model.safetensors
Checkpoint identity verified: run_name=koopman_ablation, use_dynamics_embedding=False
Loaded and wrapped ablation_100k: C:/Users/user/Downloads/panda-100k-ablation-checkpoint/model.safetensors


In [2]:
# ============================================================
# CELL 2 -- ROLLOUT-TRAJECTORY HARNESS
# Unlike the standard evaluate() harness elsewhere in this project (which
# returns MAE only), G1 needs the actual forecasted trajectory values for
# correlation-dimension computation. panda_forecast/chronos_forecast below
# are the same underlying calls as the standard harness, just returning the
# raw forecast array instead of an error metric.
# ============================================================
CONTEXT_LEN = 512
TRAIN_H = 128

def instance_norm_window(x_CT):
    mu = x_CT.mean(axis=1, keepdims=True)
    std_raw = x_CT.std(axis=1, keepdims=True)
    std = np.where(std_raw < 1e-6, 1.0, std_raw)  # degenerate-channel guard, per B3c
    return (x_CT - mu) / std, mu, std

def panda_forecast_traj(model, context_np, horizon):
    """Returns (C, horizon) forecast, autoregressive rollout for H > 128."""
    remaining = horizon
    ctx = context_np.copy()
    preds = []
    while remaining > 0:
        h = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = model.predict(context_t, h, limit_prediction_length=False, sliding_context=True)
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)

def chronos_forecast_traj(context_np, horizon):
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(ctx, prediction_length=horizon, num_samples=1)
    return out[:, 0, :].cpu().numpy()

print("Rollout harness defined.")


Rollout harness defined.


## Part 1: Correlation-Dimension Estimator + Validation Gates

Grassberger-Procaccia algorithm: Takens delay embedding at increasing
embedding dimension $m$, correlation sum $C(r)$ over log-spaced radii, slope
of $\log C(r)$ vs $\log r$ in the scaling region gives the dimension estimate
at that $m$. **The differentiator between a real low-dimensional attractor
and noise is saturation**: for a genuine chaotic system, the estimate
converges to a constant as $m$ increases; for noise, it keeps growing
roughly linearly with $m$, never saturating.

**Pre-registered gates (fixed before running):**
- Gate A (limit cycle / sine): known $d=1$. Pass if estimate saturates within
  $\pm 0.3$ of 1.0 by $m=5$.
- Gate B (Lorenz): literature value $d \approx 2.05$. Pass if estimate
  saturates within $\pm 0.5$ by $m=6$ (looser tolerance, matching this
  project's own precedent that Lorenz dimension estimation is harder than
  simpler signals).
- Gate C (white noise, negative control): should NOT saturate. Pass if the
  estimate at $m=6$ exceeds the estimate at $m=2$ by at least 1.5 (i.e.
  still growing, not converged).

**All three gates must pass before Part 2 runs.**


In [3]:
# ============================================================
# CELL 3 -- GP CORRELATION DIMENSION ESTIMATOR
# ============================================================

# --- Robust tau selection, added after inspecting this project's retired
# correlation-dimension attempt (topology_analysis.csv), which used a fixed
# tau=2 with no selection logic. This is the same failure class Section 10's
# v1 gate diagnosed for a different estimator ("MI first-strict-local-minimum
# rule is brittle to estimation noise"). Matches Section 10's v2 fix: smoothed
# MI curve, minimum required to hold over a window, autocorrelation-zero
# fallback if no clean MI minimum is found. ---

def _mutual_information(x, y, n_bins=16):
    c_xy, _, _ = np.histogram2d(x, y, bins=n_bins)
    p_xy = c_xy / c_xy.sum()
    p_x = p_xy.sum(axis=1)
    p_y = p_xy.sum(axis=0)
    p_x_p_y = np.outer(p_x, p_y)
    nz = p_xy > 0
    return float(np.sum(p_xy[nz] * np.log(p_xy[nz] / p_x_p_y[nz])))

def _tau_from_autocorr(x_1d, max_tau):
    x = x_1d - x_1d.mean()
    denom = np.dot(x, x)
    autocorr = np.array([np.dot(x[:-t], x[t:]) / denom if t > 0 else 1.0
                          for t in range(1, max_tau + 1)])
    zero_cross = np.where(np.diff(np.sign(autocorr)))[0]
    return int(zero_cross[0]) + 1 if len(zero_cross) > 0 else 1

def select_tau(x_1d, max_tau=50, n_bins=16, smooth_window=3, hold_window=2):
    """Returns a tau selected via smoothed-MI-minimum, falling back to
    autocorrelation zero-crossing. NOT validated against a known-tau ground
    truth signal -- unlike the correlation-dimension estimator itself (Gates
    A/B/C below), this selection procedure has no equivalent gate of its own.
    Flagged as a residual risk, not a fully closed one."""
    max_tau = min(max_tau, len(x_1d) // 4)
    if max_tau < smooth_window + hold_window + 2:
        return _tau_from_autocorr(x_1d, max(max_tau, 5))

    mi_vals = np.array([_mutual_information(x_1d[:-t], x_1d[t:], n_bins=n_bins)
                         for t in range(1, max_tau + 1)])
    kernel = np.ones(smooth_window) / smooth_window
    mi_smooth = np.convolve(mi_vals, kernel, mode='valid')

    for i in range(1, len(mi_smooth) - hold_window):
        if mi_smooth[i] < mi_smooth[i - 1] and all(
            mi_smooth[i] <= mi_smooth[i + j] for j in range(1, hold_window + 1)
        ):
            return i + 1  # offset: mi_smooth index 0 corresponds to tau=1

    return _tau_from_autocorr(x_1d, max_tau)


def takens_embed(x_1d, m, tau=1):
    n = len(x_1d) - (m - 1) * tau
    if n <= 0:
        raise ValueError(f"Signal too short for embedding dimension m={m}, tau={tau}")
    return np.array([x_1d[i : i + m * tau : tau] for i in range(n)])

def correlation_dimension(x_1d, m, tau=None, theiler_window=None, n_r=30):
    """Returns estimated correlation dimension at embedding dimension m.
    tau=None (default) triggers auto-selection via select_tau(); pass an
    explicit int to override (e.g. for a fixed-tau sensitivity check)."""
    if tau is None:
        tau = select_tau(x_1d)

    embedded = takens_embed(x_1d, m, tau)
    n_pts = embedded.shape[0]
    if theiler_window is None:
        theiler_window = max(int(0.02 * n_pts), 10)  # matches this project's Rosenstein convention

    dists = squareform(pdist(embedded))
    # exclude temporally close pairs (Theiler window)
    idx = np.arange(n_pts)
    close_mask = np.abs(idx[:, None] - idx[None, :]) < theiler_window
    dists_masked = dists.copy()
    dists_masked[close_mask] = np.inf
    flat_dists = dists_masked[np.triu_indices(n_pts, k=1)]
    flat_dists = flat_dists[np.isfinite(flat_dists)]

    if len(flat_dists) < 50:
        return np.nan  # not enough valid pairs after Theiler exclusion

    d_min, d_max = np.percentile(flat_dists, [1, 99])
    r_values = np.logspace(np.log10(d_min + 1e-12), np.log10(d_max), n_r)

    log_r, log_C = [], []
    for r in r_values:
        c_r = np.mean(flat_dists < r)
        if c_r > 0:
            log_r.append(np.log(r))
            log_C.append(np.log(c_r))
    if len(log_r) < 5:
        return np.nan

    log_r, log_C = np.array(log_r), np.array(log_C)
    # scaling region: middle portion of the range, excluding extremes
    lo, hi = np.percentile(log_r, [20, 80])
    mask = (log_r >= lo) & (log_r <= hi)
    if mask.sum() < 3:
        mask = np.ones_like(log_r, dtype=bool)
    slope, _ = np.polyfit(log_r[mask], log_C[mask], deg=1)
    return slope

def correlation_dimension_sweep(x_1d, m_values=range(1, 7), tau=None):
    """tau=None: auto-select ONCE from the signal, reuse across all m in the
    sweep (consistent with standard GP practice -- tau characterizes the
    signal, not the embedding dimension)."""
    if tau is None:
        tau = select_tau(x_1d)
    return {m: correlation_dimension(x_1d, m, tau=tau) for m in m_values}

print("GP estimator defined (with auto tau-selection).")


GP estimator defined (with auto tau-selection).


In [4]:
# ============================================================
# CELL 4 -- VALIDATION GATES (must pass before Part 2)
# ============================================================
print("="*70)
print("GATE A: limit cycle (sine), known d=1")
print("="*70)
t = np.linspace(0, 200, 4000)
sine_signal = np.sin(t)
sine_tau = select_tau(sine_signal)
print(f"  auto-selected tau={sine_tau}")
sine_sweep = correlation_dimension_sweep(sine_signal, m_values=range(1, 6), tau=sine_tau)
for m, d in sine_sweep.items():
    print(f"  m={m}: d_hat={d:.3f}")
gate_a_estimate = sine_sweep[5]
gate_a_pass = abs(gate_a_estimate - 1.0) <= 0.3
print(f"Gate A: d_hat(m=5)={gate_a_estimate:.3f}, target 1.0 +/- 0.3 -> {'PASS' if gate_a_pass else 'FAIL'}")

print()
print("="*70)
print("GATE B: Lorenz, literature d ~= 2.05")
print("="*70)
def simulate_lorenz_for_gate(n=6000, dt=0.01, sigma=10, rho=28, beta=8/3):
    x, y, z = 0.1, 0.0, 0.0
    xs = [x]
    for _ in range(n - 1):
        dx = sigma*(y-x); dy = x*(rho-z)-y; dz = x*y-beta*z
        x += dt*dx; y += dt*dy; z += dt*dz
        xs.append(x)
    return np.array(xs)

lorenz_x = simulate_lorenz_for_gate()[1000:]  # discard transient
lorenz_tau = select_tau(lorenz_x)
print(f"  auto-selected tau={lorenz_tau}")
lorenz_sweep = correlation_dimension_sweep(lorenz_x, m_values=range(1, 7), tau=lorenz_tau)
for m, d in lorenz_sweep.items():
    print(f"  m={m}: d_hat={d:.3f}")
gate_b_estimate = lorenz_sweep[6]
gate_b_pass = abs(gate_b_estimate - 2.05) <= 0.5
print(f"Gate B: d_hat(m=6)={gate_b_estimate:.3f}, target 2.05 +/- 0.5 -> {'PASS' if gate_b_pass else 'FAIL'}")

print()
print("="*70)
print("GATE C: white noise (negative control), should NOT saturate")
print("="*70)
rng = np.random.default_rng(0)
noise_signal = rng.standard_normal(4000)
noise_tau = select_tau(noise_signal)
print(f"  auto-selected tau={noise_tau}")
noise_sweep = correlation_dimension_sweep(noise_signal, m_values=range(1, 7), tau=noise_tau)
for m, d in noise_sweep.items():
    print(f"  m={m}: d_hat={d:.3f}")
gate_c_growth = noise_sweep[6] - noise_sweep[2]
gate_c_pass = gate_c_growth >= 1.5
print(f"Gate C: d_hat(m=6)-d_hat(m=2)={gate_c_growth:.3f}, target >= 1.5 (still growing) -> {'PASS' if gate_c_pass else 'FAIL'}")

print()
print("="*70)
ALL_GATES_PASS = gate_a_pass and gate_b_pass and gate_c_pass
if ALL_GATES_PASS:
    print("ALL GATES PASS. Proceeding to Part 2 is justified.")
else:
    print("GATE FAILURE. Per Section 1.2's estimator-validation rule, Part 2 should NOT run")
    print("on real data until this is resolved. Do not proceed past this cell if any gate failed --")
    print("iterate on the estimator (embedding dimension range, Theiler window, scaling-region")
    print("selection) the way Section 10's persistent-homology work did across v1/v2/v3, rather")
    print("than pushing through with an unvalidated instrument.")


GATE A: limit cycle (sine), known d=1
  auto-selected tau=21
  m=1: d_hat=0.805
  m=2: d_hat=1.016
  m=3: d_hat=1.009
  m=4: d_hat=1.012
  m=5: d_hat=1.011
Gate A: d_hat(m=5)=1.011, target 1.0 +/- 0.3 -> PASS

GATE B: Lorenz, literature d ~= 2.05
  auto-selected tau=15
  m=1: d_hat=0.965
  m=2: d_hat=1.441
  m=3: d_hat=1.543
  m=4: d_hat=1.640
  m=5: d_hat=1.692
  m=6: d_hat=1.710
Gate B: d_hat(m=6)=1.710, target 2.05 +/- 0.5 -> PASS

GATE C: white noise (negative control), should NOT saturate
  auto-selected tau=6
  m=1: d_hat=0.974
  m=2: d_hat=1.728
  m=3: d_hat=2.274
  m=4: d_hat=2.696
  m=5: d_hat=3.047
  m=6: d_hat=3.357
Gate C: d_hat(m=6)-d_hat(m=2)=1.628, target >= 1.5 (still growing) -> PASS

ALL GATES PASS. Proceeding to Part 2 is justified.


## Part 2: Real-Data Application

**Only proceed past this point if `ALL_GATES_PASS` printed True above.**


In [5]:
# ============================================================
# CELL 5 -- PART 2a: DOUBLE PENDULUM + LORENZ RHO=10 (published checkpoints)
# ============================================================
if not ALL_GATES_PASS:
    print("[STOPPED] Gates did not pass. Skipping Part 2a.")
else:
    try:
        print("=== Part 2a: Double Pendulum + Lorenz rho=10 rollout structure ===")

        def simulate_double_pendulum(n=4000, dt=0.02, g=9.81, l1=1.0, l2=1.0, m1=1.0, m2=1.0, seed=42):
            rng = np.random.default_rng(seed)
            th1, th2 = rng.uniform(-np.pi, np.pi, 2)
            w1, w2 = 0.0, 0.0
            traj = []
            for _ in range(n):
                traj.append(th1)
                den1 = (m1+m2)*l1 - m2*l1*np.cos(th1-th2)**2
                a1 = (m2*l1*w1**2*np.sin(th1-th2)*np.cos(th1-th2) +
                      m2*g*np.sin(th2)*np.cos(th1-th2) +
                      m2*l2*w2**2*np.sin(th1-th2) - (m1+m2)*g*np.sin(th1)) / den1
                den2 = (l2/l1)*den1
                a2 = (-m2*l2*w2**2*np.sin(th1-th2)*np.cos(th1-th2) +
                      (m1+m2)*g*np.sin(th1)*np.cos(th1-th2) -
                      (m1+m2)*l1*w1**2*np.sin(th1-th2) - (m1+m2)*g*np.sin(th2)) / den2
                w1 += dt*a1; w2 += dt*a2
                th1 += dt*w1; th2 += dt*w2
            return np.array(traj, dtype=np.float32)

        def simulate_lorenz_rho(rho, n=5000, dt=0.01):
            x, y, z = 0.1, 0.0, 0.0
            xs = [x]
            for _ in range(n-1):
                dx = 10*(y-x); dy = x*(rho-z)-y; dz = x*y-(8/3)*z
                x += dt*dx; y += dt*dy; z += dt*dz
                xs.append(x)
            return np.array(xs, dtype=np.float32)

        real_systems = {
            "double_pendulum": simulate_double_pendulum()[500:],
            "lorenz_rho10": simulate_lorenz_rho(rho=10)[500:],
        }

        H = 336
        part2a_results = []
        for name, series in real_systems.items():
            data = series[None, :]  # (1, T)
            C, T = data.shape
            start = (T - CONTEXT_LEN - H) // 2
            ctx_raw = data[:, start:start+CONTEXT_LEN]
            tgt_raw = data[:, start+CONTEXT_LEN:start+CONTEXT_LEN+H]
            ctx_norm, mu, std = instance_norm_window(ctx_raw)

            panda_pred = panda_forecast_traj(panda_model_published, ctx_norm, H)
            chronos_pred = chronos_forecast_traj(ctx_norm, H)
            tgt_norm = (tgt_raw - mu) / std

            d_ground_truth = correlation_dimension(tgt_norm[0], m=5)
            d_panda = correlation_dimension(panda_pred[0], m=5)
            d_chronos = correlation_dimension(chronos_pred[0], m=5)

            print(f"  {name}: d(ground_truth)={d_ground_truth:.3f}, "
                  f"d(panda_rollout)={d_panda:.3f}, d(chronos_rollout)={d_chronos:.3f}")
            part2a_results.append({
                "system": name, "d_ground_truth": d_ground_truth,
                "d_panda": d_panda, "d_chronos": d_chronos,
                "panda_dim_error": abs(d_panda - d_ground_truth) if not np.isnan(d_panda) else np.nan,
                "chronos_dim_error": abs(d_chronos - d_ground_truth) if not np.isnan(d_chronos) else np.nan,
            })

        pd.DataFrame(part2a_results).to_csv("g1_part2a_published_checkpoints.csv", index=False)
        print("Saved g1_part2a_published_checkpoints.csv")
    except Exception as e:
        print(f"[FAILED] Part 2a: {e}")


=== Part 2a: Double Pendulum + Lorenz rho=10 rollout structure ===


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  double_pendulum: d(ground_truth)=1.820, d(panda_rollout)=1.688, d(chronos_rollout)=2.471
  lorenz_rho10: d(ground_truth)=1.935, d(panda_rollout)=1.875, d(chronos_rollout)=2.416
Saved g1_part2a_published_checkpoints.csv


In [6]:
# ============================================================
# CELL 6 -- PART 2b: ROSSLER + BURGERS(nu=1.0) + HARMONIC, H=336 (retrained checkpoints)
# Skipped automatically if baseline_100k/ablation_100k did not load (Cell 1).
# Burgers generator confirmed verbatim-match against project history (no fix needed).
# Harmonic generator is the STABLE (solve_ivp) one from Section 18's correction --
# the retracted Euler version is NOT used here.
# ============================================================
if not ALL_GATES_PASS:
    print("[STOPPED] Gates did not pass. Skipping Part 2b.")
elif baseline_100k is None or ablation_100k is None:
    print("[SKIPPED] Retrained checkpoints not loaded (CHECKPOINT_DIR empty or loading failed). "
          "Fill in CHECKPOINT_DIR in Cell 1 and rerun from there to enable this section.")
else:
    try:
        print("=== Part 2b: Rossler + Burgers(nu=1.0) + Harmonic, H=336 rollout structure ===")

        def simulate_rossler_for_gate(n_steps=4000, dt=0.05, a=0.2, b=0.2, c=5.7, seed=42):
            rng = np.random.default_rng(seed)
            def rhs(t, y):
                return [-y[1]-y[2], y[0]+a*y[1], b+y[2]*(y[0]-c)]
            ic = rng.standard_normal(3)
            sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                             t_eval=np.linspace(0, n_steps*dt, n_steps),
                             method='RK45', rtol=1e-9, atol=1e-9)
            return sol.y[0].astype(np.float32)

        # Burgers: verbatim, confirmed matching project history (fixed_burgers_results.csv
        # provenance). PCA reduction to 16 channels, matching every other Burgers usage
        # in this project.
        from scipy.fft import fft, ifft, fftfreq
        from scipy.linalg import svd as _svd

        def simulate_burgers_stable(T=1000, N_x=128, nu=1.0, seed=42):
            rng = np.random.default_rng(seed)
            dx = 2 * np.pi / N_x
            dt_diff = 0.4 * dx**2 / (2 * nu + 1e-10)
            dt_adv = 0.4 * dx
            dt = min(dt_diff, dt_adv, 0.05)
            dt_record = 0.01
            n_sub = max(1, int(np.ceil(dt_record / dt)))
            dt_act = dt_record / n_sub
            k = fftfreq(N_x, d=1.0/N_x).astype(complex)
            dealias = np.abs(k) <= N_x // 3
            L_op = -nu * k**2
            u0_hat = np.zeros(N_x, dtype=complex)
            for m in range(1, 6):
                amp = rng.standard_normal() + 1j * rng.standard_normal()
                u0_hat[m] += amp
                u0_hat[N_x - m] += np.conj(amp)
            u0_hat *= dealias
            def rhs_hat(u_hat):
                u_phys = np.real(ifft(u_hat))
                nonlin = fft(0.5 * u_phys**2) * dealias
                return L_op * u_hat - 1j * k * nonlin
            U = np.zeros((T, N_x), dtype=np.float32)
            u_hat = u0_hat.copy()
            for t in range(T):
                U[t] = np.real(ifft(u_hat)).astype(np.float32)
                for _ in range(n_sub):
                    k1 = rhs_hat(u_hat)
                    k2 = rhs_hat(u_hat + 0.5*dt_act*k1)
                    k3 = rhs_hat(u_hat + 0.5*dt_act*k2)
                    k4 = rhs_hat(u_hat + dt_act*k3)
                    u_hat = u_hat + (dt_act/6.0)*(k1+2*k2+2*k3+k4)
                    u_hat *= dealias
            return U

        def pca_reduction(U, n_components):
            U_c = U - U.mean(axis=0, keepdims=True)
            n_c = min(n_components, min(U_c.shape)-1)
            _, _, Vt = _svd(U_c, full_matrices=False)
            return (U_c @ Vt[:n_c].T).astype(np.float32)

        # Harmonic: STABLE generator (solve_ivp), NOT the retracted Euler version.
        def simulate_harmonic_stable(omega=1.0, dt=0.05, n_steps=4000, seed=42):
            rng = np.random.default_rng(seed)
            def rhs(t, y):
                x, v = y
                return [v, -omega**2 * x]
            ic = [float(rng.standard_normal()), float(rng.standard_normal())]
            sol = solve_ivp(rhs, [0, n_steps*dt], ic,
                             t_eval=np.linspace(0, n_steps*dt, n_steps),
                             method='RK45', rtol=1e-8, atol=1e-8)
            return sol.y[0].astype(np.float32)

        rossler_data = simulate_rossler_for_gate()[500:][None, :]

        burgers_U = simulate_burgers_stable(T=1000, N_x=128, nu=1.0, seed=42)
        burgers_pca = pca_reduction(burgers_U, 16)
        burgers_data = burgers_pca.T[0:1]  # first PCA channel only, matching G1's
                                             # 1D correlation-dimension scope elsewhere

        harmonic_data = simulate_harmonic_stable()[500:][None, :]

        H = 336
        systems_2b = {
            "rossler": rossler_data,
            "burgers_nu1p0": burgers_data,
            "harmonic": harmonic_data,
        }

        part2b_results = []
        for name, data in systems_2b.items():
            C, T = data.shape
            if T < CONTEXT_LEN + H:
                print(f"  [SKIP] {name}: T={T} too short for H={H}")
                continue
            start = (T - CONTEXT_LEN - H) // 2
            ctx_raw = data[:, start:start+CONTEXT_LEN]
            tgt_raw = data[:, start+CONTEXT_LEN:start+CONTEXT_LEN+H]
            ctx_norm, mu, std = instance_norm_window(ctx_raw)

            baseline_pred = panda_forecast_traj(baseline_100k, ctx_norm, H)
            ablation_pred = panda_forecast_traj(ablation_100k, ctx_norm, H)
            tgt_norm = (tgt_raw - mu) / std

            d_ground_truth = correlation_dimension(tgt_norm[0], m=5)
            d_baseline = correlation_dimension(baseline_pred[0], m=5)
            d_ablation = correlation_dimension(ablation_pred[0], m=5)

            print(f"  {name} H=336: d(ground_truth)={d_ground_truth:.3f}, "
                  f"d(baseline_rollout)={d_baseline:.3f}, d(ablation_rollout)={d_ablation:.3f}")

            part2b_results.append({
                "system": name, "d_ground_truth": d_ground_truth,
                "d_baseline": d_baseline, "d_ablation": d_ablation,
                "baseline_dim_error": abs(d_baseline - d_ground_truth) if not np.isnan(d_baseline) else np.nan,
                "ablation_dim_error": abs(d_ablation - d_ground_truth) if not np.isnan(d_ablation) else np.nan,
            })

        pd.DataFrame(part2b_results).to_csv("g1_part2b_retrained_checkpoints.csv", index=False)
        print("Saved g1_part2b_retrained_checkpoints.csv")
    except Exception as e:
        print(f"[FAILED] Part 2b: {e}")


=== Part 2b: Rossler + Burgers(nu=1.0) + Harmonic, H=336 rollout structure ===
  rossler H=336: d(ground_truth)=2.671, d(baseline_rollout)=1.958, d(ablation_rollout)=1.852
  burgers_nu1p0 H=336: d(ground_truth)=0.929, d(baseline_rollout)=2.239, d(ablation_rollout)=1.406
  harmonic H=336: d(ground_truth)=1.222, d(baseline_rollout)=1.430, d(ablation_rollout)=1.886
Saved g1_part2b_retrained_checkpoints.csv
